In [1]:
'''
Cálculo numérico: Interpolação, Extrapolação e ajustes de curvas.
Crie as seguintes funções utilizando a linguagem Python.
1. Interpolação Linear
1.2. Interpolação Polinomial
1.3. Polinômio Interpolador de Lagrange
1.4. Polinômio Interpolador de Newton
2. Extrapolação e regressão numérica
3. Ajuste de curvas e o método dos mínimos quadrados
'''

"""
Cálculo Numérico: Interpolação, Extrapolação e Ajuste de Curvas
================================================================
Implementação completa em Python puro (sem scipy para os algoritmos core).
"""

import math
from typing import List, Tuple, Optional


# ─────────────────────────────────────────────────────────────────────────────
# 1. INTERPOLAÇÃO LINEAR
# ─────────────────────────────────────────────────────────────────────────────

def interpolacao_linear(
    x0: float, y0: float,
    x1: float, y1: float,
    x: float
) -> dict:
    """
    Interpolação linear entre dois pontos (x0, y0) e (x1, y1).

    Fórmula:
        y = y0 + (y1 - y0) / (x1 - x0) * (x - x0)

    Parâmetros:
        x0, y0: primeiro ponto conhecido
        x1, y1: segundo ponto conhecido
        x     : valor onde se deseja interpolar

    Retorna:
        dict com 'y' (resultado), 'passos' (lista descritiva) e 'n_passos'.
    """
    passos = []

    passos.append(f"Passo 1 – Calcular a inclinação (slope): m = (y1 - y0) / (x1 - x0)")
    m = (y1 - y0) / (x1 - x0)
    passos.append(f"         m = ({y1} - {y0}) / ({x1} - {x0}) = {m}")

    passos.append(f"Passo 2 – Aplicar a fórmula de interpolação: y = y0 + m * (x - x0)")
    y = y0 + m * (x - x0)
    passos.append(f"         y = {y0} + {m} * ({x} - {x0}) = {y}")

    return {"y": y, "passos": passos, "n_passos": len(passos)}


# ─────────────────────────────────────────────────────────────────────────────
# 1.2 INTERPOLAÇÃO POLINOMIAL (Vandermonde / sistema linear)
# ─────────────────────────────────────────────────────────────────────────────

def _gauss_elimination(A: List[List[float]], b: List[float]) -> List[float]:
    """Eliminação de Gauss com pivotamento parcial. Retorna solução x."""
    n = len(b)
    # Augmented matrix
    M = [A[i][:] + [b[i]] for i in range(n)]

    for col in range(n):
        # Pivotamento parcial
        max_row = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[max_row] = M[max_row], M[col]

        pivot = M[col][col]
        if abs(pivot) < 1e-12:
            raise ValueError("Sistema singular — pontos x podem ser duplicados.")

        for row in range(col + 1, n):
            factor = M[row][col] / pivot
            for j in range(col, n + 1):
                M[row][j] -= factor * M[col][j]

    # Substituição retroativa
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = M[i][n]
        for j in range(i + 1, n):
            x[i] -= M[i][j] * x[j]
        x[i] /= M[i][i]

    return x


def interpolacao_polinomial(
    xs: List[float],
    ys: List[float],
    x: float
) -> dict:
    """
    Interpolação polinomial via matriz de Vandermonde.

    Dado n+1 pontos, determina o polinômio de grau n:
        P(x) = a0 + a1*x + a2*x² + ... + an*xⁿ

    Parâmetros:
        xs: lista de abscissas conhecidas (sem repetição)
        ys: lista de ordenadas correspondentes
        x : valor onde avaliar P(x)

    Retorna:
        dict com 'y', 'coeficientes' (a0..an), 'passos', 'n_passos'.
    """
    n = len(xs)
    passos = []

    passos.append(f"Passo 1 – Montar a matriz de Vandermonde {n}×{n}.")
    V = [[xi**j for j in range(n)] for xi in xs]
    passos.append(f"         V =\n" + "\n".join(
        ["         " + str([round(v, 6) for v in row]) for row in V]
    ))

    passos.append(f"Passo 2 – Resolver V·a = y via eliminação de Gauss.")
    coefs = _gauss_elimination(V, list(ys))
    passos.append(f"         Coeficientes a = {[round(c, 8) for c in coefs]}")

    passos.append(f"Passo 3 – Avaliar P({x}) = Σ aᵢ·x^i")
    y = sum(coefs[i] * x**i for i in range(n))
    passos.append(f"         P({x}) = {y}")

    return {"y": y, "coeficientes": coefs, "passos": passos, "n_passos": len(passos)}


# ─────────────────────────────────────────────────────────────────────────────
# 1.3 POLINÔMIO INTERPOLADOR DE LAGRANGE
# ─────────────────────────────────────────────────────────────────────────────

def lagrange(
    xs: List[float],
    ys: List[float],
    x: float
) -> dict:
    """
    Interpolação de Lagrange.

    P(x) = Σ yᵢ · Lᵢ(x)
    onde  Lᵢ(x) = Π_{j≠i} (x - xⱼ) / (xᵢ - xⱼ)

    Parâmetros:
        xs, ys: pontos conhecidos
        x     : ponto de avaliação

    Retorna:
        dict com 'y', 'L_valores' (cada base avaliada), 'passos', 'n_passos'.
    """
    n = len(xs)
    passos = []
    L_vals = []

    for i in range(n):
        passos.append(f"Passo {i+1} – Calcular base de Lagrange L_{i}({x})")
        num = 1.0
        den = 1.0
        for j in range(n):
            if j != i:
                num *= (x - xs[j])
                den *= (xs[i] - xs[j])
        Li = num / den
        L_vals.append(Li)
        passos.append(f"         L_{i}({x}) = {Li:.8f}  →  contribuição: {ys[i]} × {Li:.8f} = {ys[i]*Li:.8f}")

    passos.append(f"Passo {n+1} – Somar todas as contribuições: P({x}) = Σ yᵢ·Lᵢ({x})")
    y = sum(ys[i] * L_vals[i] for i in range(n))
    passos.append(f"         P({x}) = {y}")

    return {"y": y, "L_valores": L_vals, "passos": passos, "n_passos": len(passos)}


# ─────────────────────────────────────────────────────────────────────────────
# 1.4 POLINÔMIO INTERPOLADOR DE NEWTON (Diferenças Divididas)
# ─────────────────────────────────────────────────────────────────────────────

def newton_diferencias_divididas(
    xs: List[float],
    ys: List[float],
    x: float
) -> dict:
    """
    Interpolação de Newton por diferenças divididas.

    P(x) = f[x0] + f[x0,x1](x-x0) + f[x0,x1,x2](x-x0)(x-x1) + ...

    Parâmetros:
        xs, ys: pontos conhecidos
        x     : ponto de avaliação

    Retorna:
        dict com 'y', 'tabela_dd' (tabela de diferenças divididas),
                 'coeficientes', 'passos', 'n_passos'.
    """
    n = len(xs)
    passos = []

    # Construir tabela de diferenças divididas
    passos.append("Passo 1 – Construir a tabela de diferenças divididas.")
    dd = [list(ys)]  # dd[0] = f[xi]

    for order in range(1, n):
        prev = dd[order - 1]
        curr = []
        for i in range(n - order):
            val = (prev[i + 1] - prev[i]) / (xs[i + order] - xs[i])
            curr.append(val)
        dd.append(curr)
        passos.append(f"         Ordem {order}: {[round(v, 8) for v in curr]}")

    coefs = [dd[k][0] for k in range(n)]   # diagonal principal
    passos.append(f"Passo 2 – Coeficientes (diagonal): {[round(c, 8) for c in coefs]}")

    # Horner para avaliar P(x)
    passos.append(f"Passo 3 – Avaliar P({x}) pelo método de Horner.")
    y = coefs[-1]
    for k in range(n - 2, -1, -1):
        y = y * (x - xs[k]) + coefs[k]
        passos.append(f"         k={k}: acumulado = {y}")

    return {
        "y": y,
        "tabela_dd": dd,
        "coeficientes": coefs,
        "passos": passos,
        "n_passos": len(passos),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 2. EXTRAPOLAÇÃO E REGRESSÃO LINEAR
# ─────────────────────────────────────────────────────────────────────────────

def regressao_linear(
    xs: List[float],
    ys: List[float],
    x_extrapolar: Optional[float] = None
) -> dict:
    """
    Regressão linear (mínimos quadrados): y = a + b·x

    Fórmulas:
        b = (n·Σxy - Σx·Σy) / (n·Σx² - (Σx)²)
        a = (Σy - b·Σx) / n
        R² = 1 - SS_res / SS_tot

    Parâmetros:
        xs, ys        : dados
        x_extrapolar  : se fornecido, calcula y para esse ponto (pode ser extrapolação)

    Retorna:
        dict com 'a', 'b', 'R2', 'y_extrapolar', 'passos', 'n_passos'.
    """
    n = len(xs)
    passos = []

    sx  = sum(xs)
    sy  = sum(ys)
    sxy = sum(xs[i] * ys[i] for i in range(n))
    sx2 = sum(xi**2 for xi in xs)

    passos.append(f"Passo 1 – Calcular somas: Σx={sx}, Σy={sy}, Σxy={sxy}, Σx²={sx2}, n={n}")

    denom = n * sx2 - sx**2
    passos.append(f"Passo 2 – Denominador: n·Σx² - (Σx)² = {denom}")
    if abs(denom) < 1e-12:
        raise ValueError("Denominador nulo — pontos x são todos iguais.")

    b = (n * sxy - sx * sy) / denom
    a = (sy - b * sx) / n
    passos.append(f"Passo 3 – b = (n·Σxy - Σx·Σy) / denom = {b:.8f}")
    passos.append(f"Passo 4 – a = (Σy - b·Σx) / n = {a:.8f}")

    # R²
    y_mean = sy / n
    ss_tot = sum((yi - y_mean)**2 for yi in ys)
    ss_res = sum((ys[i] - (a + b * xs[i]))**2 for i in range(n))
    R2 = 1 - ss_res / ss_tot if ss_tot > 1e-12 else 1.0
    passos.append(f"Passo 5 – R² = 1 - SS_res/SS_tot = {R2:.6f}")

    y_extra = None
    if x_extrapolar is not None:
        y_extra = a + b * x_extrapolar
        tipo = "interpolação" if min(xs) <= x_extrapolar <= max(xs) else "extrapolação"
        passos.append(f"Passo 6 – {tipo.capitalize()} em x={x_extrapolar}: y = {y_extra:.8f}")

    return {
        "a": a, "b": b, "R2": R2,
        "y_extrapolar": y_extra,
        "equacao": f"y = {a:.6f} + {b:.6f}·x",
        "passos": passos,
        "n_passos": len(passos),
    }


def regressao_polinomial(
    xs: List[float],
    ys: List[float],
    grau: int,
    x_extrapolar: Optional[float] = None
) -> dict:
    """
    Regressão polinomial de grau `grau` via mínimos quadrados.
    Resolve o sistema normal: (AᵀA)·c = Aᵀ·y

    Parâmetros:
        xs, ys       : dados
        grau         : grau do polinômio ajustado
        x_extrapolar : ponto para previsão (opcional)

    Retorna:
        dict com 'coeficientes', 'R2', 'y_extrapolar', 'passos', 'n_passos'.
    """
    n = len(xs)
    m = grau + 1        # número de coeficientes
    passos = []

    passos.append(f"Passo 1 – Montar matriz de projeto A ({n}×{m}).")
    A = [[xs[i]**j for j in range(m)] for i in range(n)]

    # AᵀA e Aᵀy
    passos.append("Passo 2 – Calcular AᵀA e Aᵀy.")
    AtA = [[sum(A[k][r] * A[k][c] for k in range(n)) for c in range(m)] for r in range(m)]
    Aty = [sum(A[k][j] * ys[k] for k in range(n)) for j in range(m)]

    passos.append("Passo 3 – Resolver sistema normal via eliminação de Gauss.")
    coefs = _gauss_elimination(AtA, Aty)
    passos.append(f"         Coeficientes: {[round(c, 8) for c in coefs]}")

    def poly_eval(xv):
        return sum(coefs[j] * xv**j for j in range(m))

    # R²
    y_mean = sum(ys) / n
    ss_tot = sum((yi - y_mean)**2 for yi in ys)
    ss_res = sum((ys[i] - poly_eval(xs[i]))**2 for i in range(n))
    R2 = 1 - ss_res / ss_tot if ss_tot > 1e-12 else 1.0
    passos.append(f"Passo 4 – R² = {R2:.6f}")

    y_extra = None
    if x_extrapolar is not None:
        y_extra = poly_eval(x_extrapolar)
        passos.append(f"Passo 5 – Previsão em x={x_extrapolar}: y = {y_extra:.8f}")

    return {
        "coeficientes": coefs,
        "R2": R2,
        "y_extrapolar": y_extra,
        "passos": passos,
        "n_passos": len(passos),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 3. AJUSTE DE CURVAS — MÍNIMOS QUADRADOS (Linearizações)
# ─────────────────────────────────────────────────────────────────────────────

def ajuste_exponencial(
    xs: List[float],
    ys: List[float],
    x_pred: Optional[float] = None
) -> dict:
    """
    Ajuste exponencial: y = A·e^(B·x)
    Linearização: ln(y) = ln(A) + B·x  →  regressão linear em (x, ln(y))

    Retorna coeficientes A, B e R² no espaço linearizado.
    """
    ln_ys = [math.log(yi) for yi in ys]
    res = regressao_linear(xs, ln_ys, math.log(x_pred) if x_pred else None)

    A = math.exp(res["a"])
    B = res["b"]

    y_pred = None
    if x_pred is not None:
        y_pred = A * math.exp(B * x_pred)

    return {
        "A": A, "B": B,
        "equacao": f"y = {A:.6f} · e^({B:.6f}·x)",
        "R2_linearizado": res["R2"],
        "y_pred": y_pred,
        "passos": res["passos"],
        "n_passos": res["n_passos"],
    }


def ajuste_potencia(
    xs: List[float],
    ys: List[float],
    x_pred: Optional[float] = None
) -> dict:
    """
    Ajuste por lei de potência: y = A·x^B
    Linearização: ln(y) = ln(A) + B·ln(x)
    """
    ln_xs = [math.log(xi) for xi in xs]
    ln_ys = [math.log(yi) for yi in ys]
    res = regressao_linear(ln_xs, ln_ys)

    A = math.exp(res["a"])
    B = res["b"]

    y_pred = None
    if x_pred is not None:
        y_pred = A * x_pred**B

    return {
        "A": A, "B": B,
        "equacao": f"y = {A:.6f} · x^{B:.6f}",
        "R2_linearizado": res["R2"],
        "y_pred": y_pred,
        "passos": res["passos"],
        "n_passos": res["n_passos"],
    }


def minimos_quadrados_geral(
    xs: List[float],
    ys: List[float],
    funcoes_base: List,   # lista de callables φ_i(x)
    x_pred: Optional[float] = None
) -> dict:
    """
    Mínimos Quadrados Generalizado para base arbitrária {φ₀, φ₁, ..., φₘ}.

    Resolve: (GᵀG)·c = Gᵀy
    onde G[i][j] = φⱼ(xᵢ)

    Parâmetros:
        xs, ys        : dados
        funcoes_base  : lista de funções base φⱼ(x)
        x_pred        : ponto para previsão (opcional)

    Exemplo de uso:
        funcoes_base = [lambda x: 1, lambda x: x, lambda x: math.sin(x)]

    Retorna:
        dict com 'coeficientes', 'R2', 'y_pred', 'passos', 'n_passos'.
    """
    n = len(xs)
    m = len(funcoes_base)
    passos = []

    passos.append(f"Passo 1 – Montar matriz G ({n}×{m}) com G[i][j] = φⱼ(xᵢ).")
    G = [[funcoes_base[j](xs[i]) for j in range(m)] for i in range(n)]

    passos.append("Passo 2 – Calcular GᵀG e Gᵀy.")
    GtG = [[sum(G[k][r] * G[k][c] for k in range(n)) for c in range(m)] for r in range(m)]
    Gty = [sum(G[k][j] * ys[k] for k in range(n)) for j in range(m)]

    passos.append("Passo 3 – Resolver sistema normal via eliminação de Gauss.")
    coefs = _gauss_elimination(GtG, Gty)
    passos.append(f"         Coeficientes: {[round(c, 8) for c in coefs]}")

    def modelo(xv):
        return sum(coefs[j] * funcoes_base[j](xv) for j in range(m))

    y_mean = sum(ys) / n
    ss_tot = sum((yi - y_mean)**2 for yi in ys)
    ss_res = sum((ys[i] - modelo(xs[i]))**2 for i in range(n))
    R2 = 1 - ss_res / ss_tot if ss_tot > 1e-12 else 1.0
    passos.append(f"Passo 4 – R² = {R2:.6f}")

    y_pred = None
    if x_pred is not None:
        y_pred = modelo(x_pred)
        passos.append(f"Passo 5 – Previsão em x={x_pred}: y = {y_pred:.8f}")

    return {
        "coeficientes": coefs,
        "R2": R2,
        "y_pred": y_pred,
        "passos": passos,
        "n_passos": len(passos),
    }


# ─────────────────────────────────────────────────────────────────────────────
# DEMONSTRAÇÃO
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    sep = "=" * 65

    # ── 1. Interpolação Linear ──────────────────────────────────────
    print(sep)
    print("1. INTERPOLAÇÃO LINEAR")
    print(sep)
    res = interpolacao_linear(1, 2, 4, 8, 2.5)
    for p in res["passos"]: print(p)
    print(f"→ y = {res['y']}   |  passos = {res['n_passos']}\n")

    # ── 1.2 Interpolação Polinomial ─────────────────────────────────
    print(sep)
    print("1.2 INTERPOLAÇÃO POLINOMIAL (Vandermonde)")
    print(sep)
    xs = [0, 1, 2, 3]
    ys_d = [1, 4, 9, 16]   # y = (x+1)²
    res = interpolacao_polinomial(xs, ys_d, 1.5)
    for p in res["passos"]: print(p)
    print(f"→ y(1.5) = {res['y']:.6f}   |  passos = {res['n_passos']}\n")

    # ── 1.3 Lagrange ────────────────────────────────────────────────
    print(sep)
    print("1.3 POLINÔMIO DE LAGRANGE")
    print(sep)
    res = lagrange(xs, ys_d, 1.5)
    for p in res["passos"]: print(p)
    print(f"→ P(1.5) = {res['y']:.6f}   |  passos = {res['n_passos']}\n")

    # ── 1.4 Newton ──────────────────────────────────────────────────
    print(sep)
    print("1.4 POLINÔMIO DE NEWTON (Diferenças Divididas)")
    print(sep)
    res = newton_diferencias_divididas(xs, ys_d, 1.5)
    for p in res["passos"]: print(p)
    print(f"→ P(1.5) = {res['y']:.6f}   |  passos = {res['n_passos']}\n")

    # ── 2. Regressão Linear + Extrapolação ──────────────────────────
    print(sep)
    print("2. REGRESSÃO LINEAR + EXTRAPOLAÇÃO")
    print(sep)
    xs2 = [1, 2, 3, 4, 5]
    ys2 = [2.1, 3.9, 6.2, 7.8, 10.1]
    res = regressao_linear(xs2, ys2, x_extrapolar=8)
    for p in res["passos"]: print(p)
    print(f"→ {res['equacao']}  |  R²={res['R2']:.4f}  |  y(8)={res['y_extrapolar']:.4f}")
    print(f"   passos = {res['n_passos']}\n")

    # ── Regressão Polinomial grau 2 ──────────────────────────────────
    print(sep)
    print("2b. REGRESSÃO POLINOMIAL (grau 2)")
    print(sep)
    xs3 = [-2, -1, 0, 1, 2]
    ys3 = [4.1, 1.2, -0.1, 0.9, 3.8]   # aprox. y = x²
    res = regressao_polinomial(xs3, ys3, grau=2, x_extrapolar=3)
    for p in res["passos"]: print(p)
    print(f"→ coefs={[round(c,4) for c in res['coeficientes']]}  |  R²={res['R2']:.4f}  |  y(3)={res['y_extrapolar']:.4f}")
    print(f"   passos = {res['n_passos']}\n")

    # ── 3. Ajuste Exponencial ────────────────────────────────────────
    print(sep)
    print("3. AJUSTE EXPONENCIAL  y = A·e^(B·x)")
    print(sep)
    xs4 = [0, 1, 2, 3, 4]
    ys4 = [1.0, 2.72, 7.39, 20.09, 54.60]   # aprox. e^x
    res = ajuste_exponencial(xs4, ys4, x_pred=5)
    for p in res["passos"]: print(p)
    print(f"→ {res['equacao']}  |  R²={res['R2_linearizado']:.4f}  |  y(5)={res['y_pred']:.4f}")
    print(f"   passos = {res['n_passos']}\n")

    # ── Mínimos Quadrados Generalizado ───────────────────────────────
    print(sep)
    print("3b. MÍNIMOS QUADRADOS GENERALIZADO  {1, x, sin(x)}")
    print(sep)
    import math as _m
    bases = [lambda x: 1, lambda x: x, lambda x: _m.sin(x)]
    xs5 = [0, 0.5, 1.0, 1.5, 2.0]
    ys5 = [1.0, 1.6, 2.0, 2.2, 2.5]
    res = minimos_quadrados_geral(xs5, ys5, bases, x_pred=2.5)
    for p in res["passos"]: print(p)
    print(f"→ coefs={[round(c,4) for c in res['coeficientes']]}  |  R²={res['R2']:.4f}  |  y(2.5)={res['y_pred']:.4f}")
    print(f"   passos = {res['n_passos']}")

1. INTERPOLAÇÃO LINEAR
Passo 1 – Calcular a inclinação (slope): m = (y1 - y0) / (x1 - x0)
         m = (8 - 2) / (4 - 1) = 2.0
Passo 2 – Aplicar a fórmula de interpolação: y = y0 + m * (x - x0)
         y = 2 + 2.0 * (2.5 - 1) = 5.0
→ y = 5.0   |  passos = 4

1.2 INTERPOLAÇÃO POLINOMIAL (Vandermonde)
Passo 1 – Montar a matriz de Vandermonde 4×4.
         V =
         [1, 0, 0, 0]
         [1, 1, 1, 1]
         [1, 2, 4, 8]
         [1, 3, 9, 27]
Passo 2 – Resolver V·a = y via eliminação de Gauss.
         Coeficientes a = [1.0, 2.0, 1.0, 0.0]
Passo 3 – Avaliar P(1.5) = Σ aᵢ·x^i
         P(1.5) = 6.25
→ y(1.5) = 6.250000   |  passos = 6

1.3 POLINÔMIO DE LAGRANGE
Passo 1 – Calcular base de Lagrange L_0(1.5)
         L_0(1.5) = -0.06250000  →  contribuição: 1 × -0.06250000 = -0.06250000
Passo 2 – Calcular base de Lagrange L_1(1.5)
         L_1(1.5) = 0.56250000  →  contribuição: 4 × 0.56250000 = 2.25000000
Passo 3 – Calcular base de Lagrange L_2(1.5)
         L_2(1.5) = 0.56250000  →  co